In [7]:
import pandas as pd

In [8]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")


In [4]:
print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)
print("\nTraining columns:")
print(train_df.columns.tolist())
print("\nTesting columns:")
print(test_df.columns.tolist())



Training data shape: (159571, 8)
Testing data shape: (153164, 2)

Training columns:
['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Testing columns:
['id', 'comment_text']


In [ ]:
train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [6]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             159571 non-null  str  
 1   comment_text   159571 non-null  str  
 2   toxic          159571 non-null  int64
 3   severe_toxic   159571 non-null  int64
 4   obscene        159571 non-null  int64
 5   threat         159571 non-null  int64
 6   insult         159571 non-null  int64
 7   identity_hate  159571 non-null  int64
dtypes: int64(6), str(2)
memory usage: 72.2 MB


In [7]:
train_df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [83]:
test_df.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


#####  SELECT FEATURES AND TARGETS



In [9]:
target_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

train_df[target_columns].sum()

toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

In [10]:
train_df[target_columns].mean() * 100

toxic            9.584448
severe_toxic     0.999555
obscene          5.294822
threat           0.299553
insult           4.936361
identity_hate    0.880486
dtype: float64

In [11]:
# Input feature
X = train_df["comment_text"]

# Target labels
y = train_df[target_columns]


In [12]:
print(X.shape)
print(y.shape)

(159571,)
(159571, 6)


##### Clean the Text

In [13]:
import re

In [14]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [15]:
X_clean = X.apply(clean_text)

In [16]:
print(X.iloc[1])
print("\n")
print(X_clean.iloc[1])

D'aww! He matches this background colour I'm seemingly stuck with. Thanks.  (talk) 21:51, January 11, 2016 (UTC)


daww he matches this background colour im seemingly stuck with thanks talk january utc


In [17]:
from nltk.corpus import stopwords
import nltk

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [
        word for word in words
        if word not in stop_words
    ]
    return " ".join(filtered_words)

X_no_stopwords = X_clean.apply(remove_stopwords)

print("Before stopword removal:")
print(X_clean.iloc[1])

print("\nAfter stopword removal:")
print(X_no_stopwords.iloc[1])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Devendra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Before stopword removal:
daww he matches this background colour im seemingly stuck with thanks talk january utc

After stopword removal:
daww matches background colour im seemingly stuck thanks talk january utc


#### Tokenization

In [18]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_clean)

In [13]:
X_sequences = tokenizer.texts_to_sequences(X_clean)

In [14]:
print(X_sequences[0])

[639, 76, 2, 123, 127, 174, 29, 629, 4522, 11331, 1041, 83, 312, 53, 2011, 10779, 51, 6445, 16, 62, 2606, 144, 8, 2760, 34, 115, 1132, 15137, 2793, 5, 46, 55, 235, 2, 410, 31, 2, 42, 28, 142, 70, 3338, 90]


In [15]:
len(X_sequences)

159571

##### Padding

In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [17]:
X_padded = pad_sequences(
    X_sequences,
    maxlen=200,
    padding="post",
    truncating="post"
)

In [18]:
print(X_padded.shape)

(159571, 200)


#### Train/Validation Split

In [19]:
from sklearn.model_selection import train_test_split


In [20]:
X_train, X_val, y_train, y_val = train_test_split(
    X_padded,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)

Training data: (127656, 200)
Validation data: (31915, 200)


In [21]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (127656, 200)
X_val: (31915, 200)
y_train: (127656, 6)
y_val: (31915, 6)


#### Build the Deep Learning Model

In [30]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

model = Sequential([
    Input(shape=(200,)),
    
    Embedding(
        input_dim=20000,
        output_dim=128
    ),

    LSTM(64),

    Dense(32, activation="relu"),

    Dense(6, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,611,686 (9.96 MB)

 Trainable params: 2,611,686 (9.96 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
X_train.shape

(127656, 200)

##### TRAIN MODEL


In [31]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64
)

Epoch 1/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 174s 84ms/step - accuracy: 0.9657 - loss: 0.1386 - val_accuracy: 0.9930 - val_loss: 0.0725
Epoch 2/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 161s 81ms/step - accuracy: 0.9942 - loss: 0.0555 - val_accuracy: 0.9941 - val_loss: 0.0503
Epoch 3/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 148s 74ms/step - accuracy: 0.9941 - loss: 0.0458 - val_accuracy: 0.9941 - val_loss: 0.0500
Epoch 4/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 211s 106ms/step - accuracy: 0.9939 - loss: 0.0410 - val_accuracy: 0.9935 - val_loss: 0.0505
Epoch 5/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 143s 72ms/step - accuracy: 0.9936 - loss: 0.0366 - val_accuracy: 0.9888 - val_loss: 0.0542


##### Check Training Performance

In [32]:
print("Training Accuracy:", history.history["accuracy"][-1])
print("Validation Accuracy:", history.history["val_accuracy"][-1])

print("Training Loss:", history.history["loss"][-1])
print("Validation Loss:", history.history["val_loss"][-1])

Training Accuracy: 0.9936469793319702
Validation Accuracy: 0.9888453483581543
Training Loss: 0.03659706562757492
Validation Loss: 0.05422503873705864


##### GENERATE PREDICTIONS


In [33]:
## Predictions
y_pred_prob = model.predict(X_val)

998/998 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step


In [61]:
print("\nPrediction Shape:")
print(y_pred_prob.shape)



Prediction Shape:
(31915, 6)


##### INITIAL PREDICTION USING 0.5 THRESHOLD

In [37]:

y_pred = (y_pred_prob >= 0.5).astype(int)

In [38]:
y_pred.shape

(31915, 6)

##### Classification Report

In [40]:
from sklearn.metrics import classification_report

print(classification_report(
    y_val,
    y_pred,
    target_names=target_columns
))

               precision    recall  f1-score   support

        toxic       0.86      0.70      0.77      3056
 severe_toxic       0.54      0.33      0.41       321
      obscene       0.84      0.76      0.80      1715
       threat       0.00      0.00      0.00        74
       insult       0.75      0.62      0.68      1614
identity_hate       0.00      0.00      0.00       294

    micro avg       0.82      0.64      0.72      7074
    macro avg       0.50      0.40      0.44      7074
 weighted avg       0.77      0.64      0.70      7074
  samples avg       0.06      0.06      0.06      7074



c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

In [41]:
y_pred_prob >= 0.5

array([[False, False, False, False, False, False],
       [False, False, False, False, False, False],
       [False, False, False, False, False, False],
       ...,
       [False, False, False, False, False, False],
       [ True, False, False, False,  True, False],
       [False, False, False, False, False, False]], shape=(31915, 6))

##### CHECK RARE CLASS PROBABILITIES


In [42]:
print("Maximum threat probability:",
      y_pred_prob[:, 3].max())

print("Maximum identity_hate probability:",
      y_pred_prob[:, 5].max())

Maximum threat probability: 0.28054836
Maximum identity_hate probability: 0.46087322


##### THRESHOLD TUNING


In [43]:
from sklearn.metrics import f1_score

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:

    threat_pred = (y_pred_prob[:, 3] >= threshold).astype(int)
    identity_pred = (y_pred_prob[:, 5] >= threshold).astype(int)

    threat_f1 = f1_score(y_val.iloc[:, 3], threat_pred)
    identity_f1 = f1_score(y_val.iloc[:, 5], identity_pred)

    print(
        "Threshold:", threshold,
        "| Threat F1:", round(threat_f1, 3),
        "| Identity Hate F1:", round(identity_f1, 3)
    )

Threshold: 0.1 | Threat F1: 0.12 | Identity Hate F1: 0.262
Threshold: 0.2 | Threat F1: 0.067 | Identity Hate F1: 0.248
Threshold: 0.3 | Threat F1: 0.0 | Identity Hate F1: 0.091
Threshold: 0.4 | Threat F1: 0.0 | Identity Hate F1: 0.04
Threshold: 0.5 | Threat F1: 0.0 | Identity Hate F1: 0.0


#####  FINAL MODEL EVALUATION


In [45]:
from sklearn.metrics import classification_report

y_pred_final = y_pred_prob.copy()

# Apply class-specific thresholds
y_pred_final[:, 0] = (y_pred_prob[:, 0] >= 0.5).astype(int)
y_pred_final[:, 1] = (y_pred_prob[:, 1] >= 0.5).astype(int)
y_pred_final[:, 2] = (y_pred_prob[:, 2] >= 0.5).astype(int)

# Lower threshold for rare classes
y_pred_final[:, 3] = (y_pred_prob[:, 3] >= 0.1).astype(int)
y_pred_final[:, 4] = (y_pred_prob[:, 4] >= 0.5).astype(int)
y_pred_final[:, 5] = (y_pred_prob[:, 5] >= 0.1).astype(int)

print(classification_report(
    y_val,
    y_pred_final,
    target_names=target_columns
))

               precision    recall  f1-score   support

        toxic       0.86      0.70      0.77      3056
 severe_toxic       0.54      0.33      0.41       321
      obscene       0.84      0.76      0.80      1715
       threat       0.10      0.15      0.12        74
       insult       0.75      0.62      0.68      1614
identity_hate       0.17      0.55      0.26       294

    micro avg       0.72      0.67      0.69      7074
    macro avg       0.55      0.52      0.51      7074
 weighted avg       0.78      0.67      0.71      7074
  samples avg       0.06      0.06      0.05      7074



c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{met

##### SAVE TRAINED MODEL & tokenizer

In [46]:
model.save("toxicity_lstm_model.keras")

print("Model saved successfully!")

Model saved successfully!


In [47]:
import pickle

with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

print("Tokenizer saved successfully!")

Tokenizer saved successfully!


#### CREATE TEXT PREDICTION FUNCTION


In [48]:
def predict_toxicity(comment):

    # Clean text
    comment = clean_text(comment)

    # Convert text to numbers
    sequence = tokenizer.texts_to_sequences([comment])

    # Padding
    padded = pad_sequences(
        sequence,
        maxlen=200,
        padding="post",
        truncating="post"
    )

    # Prediction
    probabilities = model.predict(padded, verbose=0)[0]

    return probabilities

##### CREATE FINAL LABEL PREDICTION FUNCTION

In [50]:
## test the prediction function

result = predict_toxicity("You are a stupid and disgusting person")

print(result)

[0.9756067  0.08495317 0.69149685 0.13667592 0.7520785  0.22067918]


In [52]:
labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

thresholds = [0.5, 0.5, 0.5, 0.1, 0.5, 0.1]

for label, probability, threshold in zip(labels, result, thresholds):
    prediction = "YES" if probability >= threshold else "NO"

    print(label, ":", prediction)

toxic : YES
severe_toxic : NO
obscene : YES
threat : YES
insult : YES
identity_hate : YES


##### CREATE TEXT PREDICTION FUNCTION

In [53]:
#vhe Final Prediction Function

def predict_labels(comment):

    probabilities = predict_toxicity(comment)

    labels = [
        "toxic",
        "severe_toxic",
        "obscene",
        "threat",
        "insult",
        "identity_hate"
    ]

    thresholds = [0.5, 0.5, 0.5, 0.1, 0.5, 0.1]

    predictions = {}

    for label, probability, threshold in zip(
        labels, probabilities, thresholds
    ):
        predictions[label] = (
            "YES" if probability >= threshold else "NO"
        )

    return predictions

##### TEST THE MODEL


In [ ]:
test_comment = "You are a stupid and disgusting person"

result = predict_toxicity(test_comment)

print("\nPrediction Probabilities:")
print(result)



Prediction Probabilities:
[0.9756067  0.08495317 0.69149685 0.13667592 0.7520785  0.22067918]


##### DISPLAY FINAL PREDICTIONS

In [58]:
final_result = predict_labels(test_comment)

print("\nFinal Predictions:")

for label, prediction in final_result.items():

    print(
        label,
        ":",
        prediction
    )




Final Predictions:
toxic : YES
severe_toxic : NO
obscene : YES
threat : YES
insult : YES
identity_hate : YES


In [65]:
t = "daww he matches this background colour im seemingly stuck with thanks talk january utc"

result = predict_toxicity(t)

print("\nPrediction Probabilities:")
print(result)



Prediction Probabilities:
[1.1352521e-04 7.3291582e-09 3.5905749e-05 9.6819099e-07 2.3849112e-05
 7.6802517e-06]


In [66]:
final_result = predict_labels(t)

print("\nFinal Predictions:")

for label, prediction in final_result.items():

    print(
        label,
        ":",
        prediction
    )



Final Predictions:
toxic : NO
severe_toxic : NO
obscene : NO
threat : NO
insult : NO
identity_hate : NO


## FINAL PREDICTIONS ON TEST DATA



In [82]:
X_test = test_df["comment_text"]
print(X_test.shape)


(153164,)


In [71]:
X_test_clean = X_test.apply(clean_text)


In [73]:
print(X_test.iloc[1])
print("\n")
print(X_test_clean.iloc[1])

== From RfC == 

 The title is fine as it is, IMO.


from rfc the title is fine as it is imo


In [ ]:
###Convert comments into sequences by using the tokenizer
X_test_sequences = tokenizer.texts_to_sequences(X_test_clean)
print(X_test_sequences[1])

[31, 1221, 2, 391, 9, 628, 18, 12, 9, 2637]


In [ ]:
#### padding the sequences to a fixed length of 200
X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen=200,
    padding="post",
    truncating="post"
)
print(X_test_padded[1])

[  31 1221    2  391    9  628   18   12    9 2637    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0 

In [ ]:
# Generate predictions

test_pred_prob = model.predict(
    X_test_padded,
    batch_size=64,
    verbose=1
)



2394/2394 ━━━━━━━━━━━━━━━━━━━━ 93s 39ms/step


In [81]:
### Check the result
print("Test Prediction Shape:", test_pred_prob.shape)

Test Prediction Shape: (153164, 6)


#### CREATE FINAL TEST PREDICTION CSV


In [85]:
# Apply our selected thresholds
import numpy as np
test_pred = np.zeros_like(test_pred_prob, dtype=int)

for i, threshold in enumerate(thresholds):
    test_pred[:, i] = (
        test_pred_prob[:, i] >= threshold
    ).astype(int)


# Create prediction DataFrame
test_predictions = pd.DataFrame(
    test_pred,
    columns=target_columns
)

# Add comment ID
test_predictions.insert(
    0,
    "id",
    test_df["id"].values
)


# Save predictions
test_predictions.to_csv(
    "test_predictions.csv",
    index=False
)


print("Test prediction file created successfully!")
print("Shape:", test_predictions.shape)

print("\nFirst 5 predictions:")
print(test_predictions.head())

Test prediction file created successfully!
Shape: (153164, 7)

First 5 predictions:
                 id  toxic  severe_toxic  obscene  threat  insult  \
0  00001cee341fdb12      1             0        1       0       1   
1  0000247867823ef7      0             0        0       0       0   
2  00013b17ad220c46      0             0        0       0       0   
3  00017563c3f7919a      0             0        0       0       0   
4  00017695ad8997eb      0             0        0       0       0   

   identity_hate  
0              0  
1              0  
2              0  
3              0  
4              0  


##### VERIFY FINAL PREDICTION FILE


In [86]:
print("File saved as: test_predictions.csv")

print("\nShape:")
print(test_predictions.shape)

print("\nColumns:")
print(test_predictions.columns.tolist())

print("\nTotal Predictions:")
print(len(test_predictions))

print("\nMissing Values:")
print(test_predictions.isnull().sum())

File saved as: test_predictions.csv

Shape:
(153164, 7)

Columns:
['id', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Total Predictions:
153164

Missing Values:
id               0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64
